# Chapter 6 — Text Data Pipeline (Practice)

Work through these exercises **after reading** `notes/ch06-text-data-pipeline.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: The text data pipeline — Dataset, DataLoader, collate_fn, masks
# MATH:  mask[b, t] = (t < length[b]);  dynamic padding to per-batch max_len
# REF:   B00 ch06 notes — text-data-pipeline
# ============================================================

# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, IterableDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

# A tiny fixed "tokenizer": split on spaces, map words to ids via a vocab.
VOCAB = {"<pad>": 0, "i": 1, "love": 2, "pytorch": 3, "tensors": 4,
         "are": 5, "hard": 6, "fun": 7, "really": 8, "very": 9}
PAD_ID = VOCAB["<pad>"]

def tokenize(text):
    """'i love pytorch' -> [1, 2, 3] (a Python list of int ids)."""
    return [VOCAB[word] for word in text.split()]

RAW_TEXTS = ["i love pytorch", "tensors are hard", "pytorch is really very fun".replace("is ", ""),
             "i love tensors"]
RAW_LABELS = [1, 0, 1, 1]
print("corpus:", list(zip(RAW_TEXTS, RAW_LABELS)))

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)
corpus: [('i love pytorch', 1), ('tensors are hard', 0), ('pytorch really very fun', 1), ('i love tensors', 1)]


## Exercise 1 — A Map-Style Text Dataset

Implement `TextDataset` with `__len__` and `__getitem__`. Each item is a tuple `(token_ids_tensor, label)` where the token IDs are an `int64` 1-D tensor produced by `tokenize`. Crucially, `__getitem__` returns **one raw, un-padded example** — no batching, no padding.

**Decision you're practicing:** map-style Dataset (`__len__` + `__getitem__`) as "how do I get example *i*?" — and that padding is deliberately *not* its job.

In [2]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)                                  # DataLoader needs this to plan batches

    def __getitem__(self, index):
        token_ids = torch.tensor(tokenize(self.texts[index]), dtype=torch.long)   # 1-D, un-padded
        return token_ids, self.labels[index]                   # ONE example, ragged by nature

dataset = TextDataset(RAW_TEXTS, RAW_LABELS)
for i in range(len(dataset)):
    ids, label = dataset[i]
    print(f"example {i}: ids={ids.tolist()} (len {len(ids)}), label={label}")

example 0: ids=[1, 2, 3] (len 3), label=1
example 1: ids=[4, 5, 6] (len 3), label=0
example 2: ids=[3, 8, 9, 7] (len 4), label=1
example 3: ids=[1, 2, 4] (len 3), label=1


**Verification**

In [3]:
# --- Verification: Exercise 1 ---
assert len(dataset) == 4, f"__len__ should be 4, got {len(dataset)}"
ids0, label0 = dataset[0]
assert isinstance(ids0, torch.Tensor) and ids0.dtype == torch.long, "token ids must be an int64 tensor"
assert ids0.tolist() == [1, 2, 3], f"example 0 should tokenize to [1,2,3], got {ids0.tolist()}"
assert label0 == 1
# examples are ragged — that is expected and correct
lengths = [len(dataset[i][0]) for i in range(len(dataset))]
assert lengths == [3, 3, 4, 3], f"raw lengths should be ragged {[3,3,4,3]}, got {lengths}"
print(f"dataset yields {len(dataset)} un-padded examples with lengths {lengths} ✓")
print("Exercise 1 passed ✓")

dataset yields 4 un-padded examples with lengths [3, 3, 4, 3] ✓
Exercise 1 passed ✓


## Exercise 2 — The Default-Collate Crash

Feed the ragged dataset to a `DataLoader` with **no** `collate_fn` and predict what happens. Then capture the error to confirm.

**Decision you're practicing:** recognizing that the default collate is `torch.stack`, which requires equal shapes — so ragged text crashes loudly, pointing straight at the need for a custom `collate_fn`.

In [4]:
prediction = "raises"     # default_collate calls torch.stack, which demands identical shapes

default_loader = DataLoader(dataset, batch_size=4)   # no collate_fn → default_collate
crash_message = None
try:
    batch = next(iter(default_loader))
    print("no crash?!", batch)
except RuntimeError as err:
    crash_message = str(err)
    print(f"RuntimeError (as predicted):\n  {crash_message}")
print("\n→ the fix is a custom collate_fn that pads before stacking (exercise 3)")

RuntimeError (as predicted):
  stack expects each tensor to be equal size, but got [3] at entry 0 and [4] at entry 2

→ the fix is a custom collate_fn that pads before stacking (exercise 3)


**Verification**

In [5]:
# --- Verification: Exercise 2 ---
assert prediction == "raises", "default collate on ragged text raises — it can't stack unequal shapes"
assert crash_message is not None, "you should have caught a RuntimeError"
assert "stack expects each tensor to be equal size" in crash_message, \
    f"expected a stack-size error, got: {crash_message}"
print("confirmed: default_collate (torch.stack) crashes loudly on ragged sequences ✓")
print("Exercise 2 passed ✓")

confirmed: default_collate (torch.stack) crashes loudly on ragged sequences ✓
Exercise 2 passed ✓


## Exercise 3 — A Padding `collate_fn`

Write `collate_batch(examples)` that takes the DataLoader's list of `(token_ids, label)` tuples and returns three tensors: `padded` `(B, max_len)` via `pad_sequence`, `labels` `(B,)`, and `lengths` `(B,)` captured **before** padding. Pad per-batch (dynamic padding), not to any global maximum.

**Decision you're practicing:** `collate_fn: list → batch` with per-batch dynamic padding — and recording true lengths before padding so masks are trivial later.

In [6]:
def collate_batch(examples):
    """examples: list of (token_ids_tensor, label).
    Returns (padded (B, max_len), labels (B,), lengths (B,))."""
    sequences = [token_ids for token_ids, _ in examples]
    labels = torch.tensor([label for _, label in examples])
    lengths = torch.tensor([len(seq) for seq in sequences])    # BEFORE padding — needed for masks
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)   # (B, max_len in THIS batch)
    return padded, labels, lengths

padded_loader = DataLoader(dataset, batch_size=4, collate_fn=collate_batch)
padded, labels, lengths = next(iter(padded_loader))
print(f"padded {tuple(padded.shape)}:\n{padded}")
print(f"labels : {labels.tolist()}")
print(f"lengths: {lengths.tolist()}   ← the pre-padding truth")

padded (4, 4):
tensor([[1, 2, 3, 0],
        [4, 5, 6, 0],
        [3, 8, 9, 7],
        [1, 2, 4, 0]])
labels : [1, 0, 1, 1]
lengths: [3, 3, 4, 3]   ← the pre-padding truth


**Verification**

In [7]:
# --- Verification: Exercise 3 ---
assert padded is not None, "fill in the stub above first"
assert padded.shape == (4, 4), f"batch should pad to (4, max_len=4), got {tuple(padded.shape)}"
assert padded.dtype == torch.long, "padded token ids must stay int64"
assert lengths.tolist() == [3, 3, 4, 3], f"lengths should be pre-padding {[3,3,4,3]}, got {lengths.tolist()}"
assert labels.tolist() == [1, 0, 1, 1]
# row 0 was [1,2,3] → padded with one PAD_ID
assert padded[0].tolist() == [1, 2, 3, PAD_ID], f"row 0 mis-padded: {padded[0].tolist()}"
# dynamic padding: a batch of only short sequences should be narrower
short_ds = TextDataset(["i love pytorch", "i love tensors"], [1, 1])
short_padded, _, _ = next(iter(DataLoader(short_ds, batch_size=2, collate_fn=collate_batch)))
assert short_padded.shape == (2, 3), f"a short-only batch should be width 3, got {tuple(short_padded.shape)}"
print(f"dynamic padding confirmed: full batch width {padded.shape[1]}, short-only batch width {short_padded.shape[1]} ✓")
print("Exercise 3 passed ✓")

dynamic padding confirmed: full batch width 4, short-only batch width 3 ✓
Exercise 3 passed ✓


## Exercise 4 — The Attention Mask, Two Ways

Build the boolean real-vs-pad mask two independent ways and prove they agree:

1. `mask_from_pad` — from the padded tensor: positions not equal to `PAD_ID`
2. `mask_from_lengths` — from the lengths vector: `arange(max_len) < length` (broadcasting from ch01)

**Decision you're practicing:** constructing the attention mask in the pipeline — the boolean record that later feeds attention masking, pooling, and the loss.

In [8]:
def mask_from_pad(padded, pad_id):
    """(B, T) padded ids -> (B, T) bool mask, True where real."""
    return padded != pad_id

def mask_from_lengths(lengths, max_len):
    """(B,) lengths -> (B, max_len) bool mask via arange broadcasting."""
    positions = torch.arange(max_len)                 # (max_len,)
    return positions[None, :] < lengths[:, None]      # (1, T) < (B, 1) → (B, T)  [ch01 broadcasting]

mask_a = mask_from_pad(padded, PAD_ID)
mask_b = mask_from_lengths(lengths, padded.shape[1])
print(f"mask from pad value:\n{mask_a}")
print(f"mask from lengths:\n{mask_b}")
print(f"identical? {torch.equal(mask_a, mask_b)}")

mask from pad value:
tensor([[ True,  True,  True, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True],
        [ True,  True,  True, False]])
mask from lengths:
tensor([[ True,  True,  True, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True],
        [ True,  True,  True, False]])
identical? True


**Verification**

In [9]:
# --- Verification: Exercise 4 ---
mask_a = mask_from_pad(padded, PAD_ID)
mask_b = mask_from_lengths(lengths, padded.shape[1])
assert mask_a is not None and mask_b is not None, "fill in both stubs first"
assert mask_a.dtype == torch.bool and mask_b.dtype == torch.bool, "masks must be bool"
assert torch.equal(mask_a, mask_b), "the two constructions must agree"
expected = torch.tensor([[True, True, True, False],
                         [True, True, True, False],
                         [True, True, True, True],
                         [True, True, True, False]])
assert torch.equal(mask_a, expected), f"mask wrong:\n{mask_a}"
assert mask_a.sum().item() == sum(lengths.tolist()), "total True count must equal total real tokens"
print(f"both constructions agree; {mask_a.sum().item()} real tokens across the batch ✓")
print("Exercise 4 passed ✓")

both constructions agree; 13 real tokens across the batch ✓
Exercise 4 passed ✓


## Exercise 5 — drop_last and Shuffle Determinism

Two predictions about how the DataLoader orders and groups examples:

1. For a 10-example dataset with `batch_size=4`, predict the batch sizes with `drop_last=False` and with `drop_last=True`.
2. Predict whether two passes of a `shuffle=True` loader **seeded identically** produce the same order.

**Decision you're practicing:** `drop_last` (uniform batches vs keeping every example) and reproducible shuffling via seeding.

In [10]:
class RangeDataset(Dataset):
    def __len__(self): return 10
    def __getitem__(self, i): return i

range_ds = RangeDataset()

batch_sizes_keep_prediction = [4, 4, 2]    # 10 = 4+4+2, the ragged tail kept
batch_sizes_drop_prediction = [4, 4]       # the final short batch dropped
same_seed_same_order_prediction = True     # same generator seed → same permutation

keep = [len(b) for b in DataLoader(range_ds, batch_size=4, drop_last=False)]
drop = [len(b) for b in DataLoader(range_ds, batch_size=4, drop_last=True)]
print(f"drop_last=False batch sizes: {keep}")
print(f"drop_last=True  batch sizes: {drop}")

gen1 = torch.Generator().manual_seed(123)
gen2 = torch.Generator().manual_seed(123)
order1 = list(iter(DataLoader(range_ds, batch_size=10, shuffle=True, generator=gen1)))[0].tolist()
order2 = list(iter(DataLoader(range_ds, batch_size=10, shuffle=True, generator=gen2)))[0].tolist()
print(f"seeded shuffle order 1: {order1}")
print(f"seeded shuffle order 2: {order2}   ← identical seed → identical order")

drop_last=False batch sizes: [4, 4, 2]
drop_last=True  batch sizes: [4, 4]
seeded shuffle order 1: [2, 1, 6, 0, 4, 7, 9, 8, 3, 5]
seeded shuffle order 2: [2, 1, 6, 0, 4, 7, 9, 8, 3, 5]   ← identical seed → identical order


**Verification**

In [11]:
# --- Verification: Exercise 5 ---
keep = [len(b) for b in DataLoader(range_ds, batch_size=4, drop_last=False)]
drop = [len(b) for b in DataLoader(range_ds, batch_size=4, drop_last=True)]
assert batch_sizes_keep_prediction == keep == [4, 4, 2], f"drop_last=False → {keep}"
assert batch_sizes_drop_prediction == drop == [4, 4], f"drop_last=True → {drop}"

gen1 = torch.Generator().manual_seed(123)
gen2 = torch.Generator().manual_seed(123)
order1 = next(iter(DataLoader(range_ds, batch_size=10, shuffle=True, generator=gen1))).tolist()
order2 = next(iter(DataLoader(range_ds, batch_size=10, shuffle=True, generator=gen2))).tolist()
assert same_seed_same_order_prediction == (order1 == order2) == True, "identical seeds must give identical order"
assert sorted(order1) == list(range(10)), "a shuffle is a permutation — every example exactly once"
print(f"drop_last predictions correct; seeded shuffle is reproducible ({order1}) ✓")
print("Exercise 5 passed ✓")

drop_last predictions correct; seeded shuffle is reproducible ([2, 1, 6, 0, 4, 7, 9, 8, 3, 5]) ✓
Exercise 5 passed ✓


## Exercise 6 — A Streaming IterableDataset

Not all data is indexable. Implement `TokenStream`, an `IterableDataset` that yields tokenized `(ids, label)` examples one at a time from a generator — no `__len__`, no random access.

**Decision you're practicing:** map-style vs iterable — reaching for `IterableDataset` when data is a stream you can't index.

In [12]:
class TokenStream(IterableDataset):
    """Streams tokenized examples one at a time (no __len__, no __getitem__)."""
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __iter__(self):
        for text, label in zip(self.texts, self.labels):
            yield torch.tensor(tokenize(text), dtype=torch.long), label     # produced lazily

stream = TokenStream(RAW_TEXTS, RAW_LABELS)
# an IterableDataset works with DataLoader too — collate still pads (shuffle is NOT allowed here)
stream_loader = DataLoader(stream, batch_size=2, collate_fn=collate_batch)
for batch_index, (padded_batch, label_batch, length_batch) in enumerate(stream_loader):
    print(f"streamed batch {batch_index}: padded {tuple(padded_batch.shape)}, labels {label_batch.tolist()}")

streamed batch 0: padded (2, 3), labels [1, 0]
streamed batch 1: padded (2, 4), labels [1, 1]


**Verification**

In [13]:
# --- Verification: Exercise 6 ---
assert isinstance(stream, IterableDataset), "TokenStream must subclass IterableDataset"
assert not hasattr(stream, "__getitem__") or type(stream).__getitem__ is IterableDataset.__getitem__, \
    "a stream should not implement random access"
collected = list(stream)
assert len(collected) == 4, f"stream should yield 4 examples, got {len(collected)}"
first_ids, first_label = collected[0]
assert first_ids.tolist() == [1, 2, 3] and first_label == 1, "first streamed example is wrong"

# it drives a DataLoader the same way a map-style dataset does
batches = list(DataLoader(stream, batch_size=2, collate_fn=collate_batch))
assert len(batches) == 2 and batches[0][0].shape[0] == 2, "streaming through the DataLoader failed"
print(f"streamed {len(collected)} examples; DataLoader made {len(batches)} padded batches ✓")
print("Exercise 6 passed ✓")

streamed 4 examples; DataLoader made 2 padded batches ✓
Exercise 6 passed ✓


## Exercise 7 — The Whole Pipeline, End to End

Assemble everything: `Dataset` → `DataLoader(collate_fn)` → a tiny model that embeds, **masked-mean-pools** (chapters 1 & 5), and classifies. Then prove the payoff of the mask: implement `masked_mean_pool` so that **padding cannot change the pooled vector**.

The verification pools the same batch twice — once at its natural width, once after gluing on extra pad columns — and demands identical results.

**Decision you're practicing:** the whole chapter composed — and *why* the mask matters: without it, pad tokens leak into the sentence representation.

In [14]:
class MeanPoolClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, padded, mask):
        embedded = self.embedding(padded)              # (B, T, D)
        pooled = masked_mean_pool(embedded, mask)      # (B, D)
        return self.head(pooled)                       # (B, num_classes) raw logits

def masked_mean_pool(embedded, mask):
    """(B, T, D) embeddings + (B, T) bool mask -> (B, D), averaging REAL tokens only."""
    mask_float = mask.unsqueeze(-1).to(embedded.dtype)             # (B, T, 1)
    summed = (embedded * mask_float).sum(dim=1)                    # (B, D) — pad rows contribute 0
    counts = mask_float.sum(dim=1).clamp(min=1)                    # (B, 1) — real-token counts, keepdim
    return summed / counts                                        # (B, D) broadcast divide

torch.manual_seed(0)
model = MeanPoolClassifier(len(VOCAB), embed_dim=8, num_classes=2, pad_id=PAD_ID)
loader = DataLoader(dataset, batch_size=4, collate_fn=collate_batch)
padded_b, labels_b, lengths_b = next(iter(loader))
mask_b = mask_from_lengths(lengths_b, padded_b.shape[1])
logits = model(padded_b, mask_b)
print(f"pipeline output logits: {tuple(logits.shape)}  # (batch, num_classes)")
print(f"one training step is now possible: loss = {F.cross_entropy(logits, labels_b).item():.4f}")

pipeline output logits: (4, 2)  # (batch, num_classes)
one training step is now possible: loss = 0.5524


**Verification**

In [15]:
# --- Verification: Exercise 7 ---
torch.manual_seed(0)
model = MeanPoolClassifier(len(VOCAB), embed_dim=8, num_classes=2, pad_id=PAD_ID)
loader = DataLoader(dataset, batch_size=4, collate_fn=collate_batch)
padded_b, labels_b, lengths_b = next(iter(loader))
mask_b = mask_from_lengths(lengths_b, padded_b.shape[1])

logits = model(padded_b, mask_b)
assert logits is not None, "implement masked_mean_pool first"
assert logits.shape == (4, 2), f"expected (4, 2) logits, got {tuple(logits.shape)}"

# THE PAYOFF: gluing on extra padding columns must not change the pooled representation
extra_pad = torch.full((4, 3), PAD_ID, dtype=torch.long)
wider_padded = torch.cat([padded_b, extra_pad], dim=1)             # (4, 7) — 3 more pad columns
wider_mask = mask_from_lengths(lengths_b, wider_padded.shape[1])   # (4, 7) mask, same real counts
embedded_narrow = model.embedding(padded_b)
embedded_wide = model.embedding(wider_padded)
pooled_narrow = masked_mean_pool(embedded_narrow, mask_b)
pooled_wide = masked_mean_pool(embedded_wide, wider_mask)
assert torch.allclose(pooled_narrow, pooled_wide, atol=1e-6), \
    "extra padding changed the pooled vector — the mask isn't fully protecting the mean!"
print("adding pad columns left the pooled vectors identical — the mask works ✓")

# a real training step end to end
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
loss_before = F.cross_entropy(model(padded_b, mask_b), labels_b)
loss_before.backward()
optimizer.step()
print(f"end-to-end step ran; loss = {loss_before.item():.4f} ✓")
print("Exercise 7 passed ✓")

adding pad columns left the pooled vectors identical — the mask works ✓


end-to-end step ran; loss = 0.5524 ✓
Exercise 7 passed ✓


---
## Done!

Compare your work against `solved/ch06-text-data-pipeline-solved.ipynb`.

The pipeline now emits exactly the batches a model consumes — padded token IDs, labels, and a mask that provably keeps padding out of the representation. Next is the **training block's** centerpiece: **ch07 — Training Loop Anatomy** — the forward/backward/step dance, in the right order, driving everything you've built across chapters 1–6.